# **Licenciatura em Ciências da Computação**

### Aprendizagem Computacional 25/26

# Preparing Data for scikit-learn (NumPy vs pandas)


## 1. Imports


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression


# 2. NumPy input


### Data


In [2]:
X = np.array([
    [23, 20000],
    [45, 50000],
    [31, 32000],
    [35, 40000],
    [52, 58000]
])

y = np.array([0, 1, 0, 1, 1])


### Split


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
X

array([[   23, 20000],
       [   45, 50000],
       [   31, 32000],
       [   35, 40000],
       [   52, 58000]])

In [ ]:
X_train

array([[   52, 58000],
       [   31, 32000],
       [   23, 20000],
       [   35, 40000]])

In [5]:
X_test

array([[   45, 50000]])

In [6]:
y_train
y_test

array([1])

### Model


In [7]:
model_np = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression())
])

model_np.fit(X_train, y_train)
model_np.predict(X_test), model_np.predict_proba(X_test)

(array([1]), array([[0.23458519, 0.76541481]]))

# 3. pandas input


### Data


In [8]:
pandas_df = pd.DataFrame({
    "age": [23, 45, 31, 35, 52],
    "salary": [20000, 50000, 32000, 40000, 58000],
    "bought": [0, 1, 0, 1, 1]
})

X_pandas = pandas_df.drop(columns="bought")
y_pandas = pandas_df["bought"]


### Split


In [9]:
X_train_pandas, X_test_pandas, y_train_pandas, y_test_pandas = train_test_split(
    X_pandas, y_pandas, test_size=0.2, random_state=42
)


In [10]:
display(X_train_pandas)

,age,salary
4,52,58000
2,31,32000
0,23,20000
3,35,40000


In [11]:
display(X_test_pandas)

,age,salary
1,45,50000


In [12]:
display(y_train_pandas)

4    1
2    0
0    0
3    1
Name: bought, dtype: int64

In [13]:
display(y_test_pandas)

1    1
Name: bought, dtype: int64

### Model


In [14]:
model_pandas = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression())
])

model_pandas.fit(X_train_pandas, y_train_pandas)
model_pandas.predict(X_test_pandas)

array([1])

## 4. Single sample prediction


### NumPy


In [15]:
model_np.predict([[30, 40000]]), model_np.predict_proba([[30, 40000]])


(array([0]), array([[0.52546882, 0.47453118]]))

### pandas


In [19]:
sample = pd.DataFrame([{
    "age": 30,
    "salary": 40000
}])

model_pandas.predict(sample), model_pandas.predict_proba(sample)


(array([0]), array([[0.52546882, 0.47453118]]))

In [20]:
model_pandas.predict([[30, 40000]]), model_pandas.predict([[40000, 30]])

/home/jt-s/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/jt-s/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


(array([0]), array([1]))

In [21]:
sample_switched_cols = pd.DataFrame([{
    "salary": 40000,
    "age": 30
}])

model_pandas.predict(sample_switched_cols), model_pandas.predict_proba(sample_switched_cols)

ValueError: The feature names should match those that were passed during fit.
Feature names must be in the same order as they were in fit.


The `ValueError` above demonstrates a crucial difference when using pandas DataFrames with `sklearn` pipelines that involve transformers like `StandardScaler`.

When `model_pandas` was fitted, it learned the order of features from `X_train_pandas` (which was `['age', 'salary']`). When `sample_switched_cols` was passed for prediction, its column order was `['salary', 'age']`.

`sklearn` with pandas DataFrames is smart enough to detect this mismatch in feature names and order, raising a `ValueError` to prevent incorrect predictions.

In contrast, with NumPy arrays, there are no feature names. If you were to switch the order of columns in a NumPy array passed for prediction, `sklearn` would process the data in the new, incorrect order without raising an error, leading to silently wrong predictions.